In [39]:
# Run this cell first!
class _CgAnswers: ...
answers = _CgAnswers()

# Analytics for a better world
## Assignment 6: Linear programming (with indexed variables)

In [40]:
# DO NOT CHANGE
import sys
at_colab = "google.colab" in sys.modules

if at_colab:
    import shutil
    if not shutil.which("pyomo"):
        !pip install -q pyomo
        assert(shutil.which("pyomo"))

# Question 1 - Larry Edison (5 points)

In [41]:
# DO NOT CHANGE wage, full_time, part_time, A_total - D, A_ratio - D,
import pyomo.environ as pyo

In [42]:
### BEGIN SOLUTION
def model_larry_edison():
  model = pyo.ConcreteModel(name='larry_edison')

  model.full_time = pyo.Var(pyo.RangeSet(1, 3), domain=pyo.NonNegativeIntegers)
  model.part_time = pyo.Var(pyo.RangeSet(1, 4), domain=pyo.NonNegativeIntegers)

  full_time = model.full_time
  part_time = model.part_time

  @model.Objective(sense=pyo.minimize)
  def wage(model):
    return 320 * sum(full_time[i] for i in full_time) + 120 * sum(part_time[j] for j in part_time)

  @model.Constraint()
  def A_total(model):
    return full_time[1] + part_time[1] >= 4
  @model.Constraint()
  def B_total(model):
    return full_time[1] + full_time[2] + part_time[2] >= 8
  @model.Constraint()
  def C_total(model):
    return full_time[3] + full_time[2] + part_time[3] >= 10
  @model.Constraint()
  def D_total(model):
    return full_time[3] + part_time[4] >= 6

  @model.Constraint()
  def A_ratio(model):
    return full_time[1]>= 2 * part_time[1]
  @model.Constraint()
  def B_ratio(model):
    return (full_time[1] + full_time[2]) >= 2 * part_time[2]
  @model.Constraint()
  def C_ratio(model):
    return (full_time[3] + full_time[2]) >= 2 * part_time[3]
  @model.Constraint()
  def D_ratio(model):
    return full_time[3] >= 2 * part_time[4]
  return model
### END SOLUTION

In [43]:
# DO NOT CHANGE
answers.Q1_obj = globals().get("model_larry_edison", Ellipsis)
answers.Q1_con = globals().get("model_larry_edison", Ellipsis)
answers.Q1_out = globals().get("model_larry_edison", Ellipsis)

# Question 2 - Oxbridge (5 points)

In [50]:
# DO NOT CHANGE
import pyomo.environ as pyo
import pandas as pd
from io import StringIO

data = StringIO('''Operator;Wage;Mon;Tue;Wed;Thu;Fri;MinHours
K.C.; 25 ; 6 ; 0 ; 6 ; 0 ; 6 ; 8
D.H.; 26 ; 0 ; 6 ; 0 ; 6 ; 0 ; 8
H.B.; 24 ; 4 ; 8 ; 4 ; 0 ; 4 ; 8
S.C.; 23 ; 5 ; 5 ; 5 ; 0 ; 5 ; 8
K.S.; 28 ; 3 ; 0 ; 3 ; 8 ; 0 ; 7
N.K.; 30 ; 0 ; 0 ; 0 ; 6 ; 2 ; 7 ''')

df = pd.read_csv(data, sep=";", index_col='Operator')
days = ['Mon','Tue','Wed','Thu','Fri']

In [60]:
### BEGIN SOLUTION
def model_oxbridge():
    model = pyo.ConcreteModel(name="oxbridge")

    # First index: operator. Second index: day.
    model.x = pyo.Var(df.index, days, domain=pyo.NonNegativeIntegers)

    @model.Constraint(df.index)
    def min_hours(model, operator):
        return sum(model.x[operator, day] for day in days) >= df.loc[operator, "MinHours"]

    @model.Constraint(df.index, days)
    def availability(model, operator, day):
        return model.x[operator, day] <= df.loc[operator, day]

    @model.Constraint(days)
    def day_total(model, day):
        return sum(model.x[operator, day] for operator in df.index) == 14

    @model.Objective(sense=pyo.minimize)
    def wage(model):
        return sum(
            df.loc[operator, "Wage"] * model.x[operator, day]
            for operator in df.index
            for day in days
        )

    return model
### END SOLUTION

In [46]:
# DO NOT CHANGE
answers.df = globals().get("df", Ellipsis)
answers.Q2_obj = globals().get("model_oxbridge", Ellipsis)
answers.Q2_con = globals().get("model_oxbridge", Ellipsis)
answers.Q2_out = globals().get("model_oxbridge", Ellipsis)